In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [2]:
from building_blocks import GPTModel

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();

In [3]:
def text_to_token(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    decoded = tokenizer.decode(flat.tolist())
    return decoded

tokenizer = tiktoken.get_encoding("gpt2")
token_to_text(text_to_token('Hi hello', tokenizer), tokenizer)

'Hi hello'

In [4]:
from building_blocks import generate_text_simple
start_context = "Hi how are"
token_ids = generate_text_simple(model=model, idx=text_to_token(start_context, tokenizer),
                                 max_new_tokens= 10, context_size=GPT_CONFIG_124M['context_length'])
token_ids.squeeze(0)

token_to_text(token_ids, tokenizer)

'Hi how are handlers Directionsolkien exhaustinggooOM minded universal Intelligent flurry'

In [5]:
#Calculate text generation loss: Cross entropy and perplxity
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [6]:
with torch.no_grad():
    logits = model(inputs)

logits.shape

probs = torch.softmax(logits, dim=1)
print(probs.shape)
out = probs.argmax(dim=2)
out


torch.Size([2, 3, 50257])


tensor([[28581, 46681, 21177],
        [34209, 32978, 16624]])

In [7]:
print(f'Target batch 1: {token_to_text(targets[0], tokenizer)}')
print(f'Target batch 2: {token_to_text(out[0], tokenizer)}')

Target batch 1:  effort moves you
Target batch 2:  182 hiber Admir


In [8]:
text_idx = 0
target_probas_1 = probs[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probs[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)


Text 1: tensor([0.5630, 0.4563, 0.2007])
Text 2: tensor([0.2317, 0.4798, 0.0796])


In [9]:
# Compute logarithm of all token probabilities
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)


tensor([-0.5745, -0.7847, -1.6059, -1.4625, -0.7344, -2.5305])


In [10]:
# Calculate the average probability for each token
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-1.2821)


In [31]:
logits.flatten(0,1).shape


torch.Size([512, 50257])

In [12]:
flat_logits = logits.flatten(0,1)
flat_target = targets.flatten()
loss = torch.nn.functional.cross_entropy(flat_logits, flat_target)
loss

tensor(10.7940)

In [13]:
# Calculating training and evalluation loss
from building_blocks import createDataLoader
with open('../../dataset/the-verdict.txt', 'r') as f:
    text_data = f.read()

train_ration = 0.9
train_quantity = int(len(text_data)*train_ration)
train_data = text_data[: train_quantity]
test_data = text_data[train_quantity: ]
print(len(train_data))
train_data_loader = createDataLoader(text=train_data, batch_size=2, max_length=256, stride=256, shuffle= True,
                     drop_last=True, num_workers=0)

test_data_loader = createDataLoader(text=test_data, batch_size=2, max_length=256, stride=256, shuffle= True,
                     drop_last=True, num_workers=0)
for input, target in train_data_loader:
    print(input.shape, target.shape)
    

18431
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


In [14]:
device = torch.device('mps' if torch.backends.mps.is_available else 'cpu')
model.to(device=device);

In [29]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    print(target_batch.shape)

    logits = model(input_batch)
    print(logits.flatten(0, 1).shape)

    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [30]:
with torch.no_grad():
    train_loss = calc_loss_loader(data_loader=train_data_loader, model=model, device=device)
    test_loss = calc_loss_loader(data_loader=test_data_loader, model=model, device=device)
    print(train_loss)
    print(test_loss)


    

torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
torch.Size([2, 256])
torch.Size([512, 50257])
0.011248226277530193
7.263232231140137


In [18]:
learning_rate = 0.004
epochs = 20

optimizer = torch.optim.AdamW(lr=0.0004, params= model.parameters() ,weight_decay=0.1)
loss_function = nn.functional.cross_entropy
model.to(device=device)

for epoch in range(epochs):
    
    for batch_inputs, batch_targets in train_data_loader:
        total_loss = 0
        # model in training mode
        model.train()

        # disable gradients
        optimizer.zero_grad()

        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        # forward pass
        logits = model(batch_inputs)
        
        #print(logits.shape)
        #print(batch_targets.shape)

        # calculate loss
        loss = loss_function(logits.flatten(0,1), batch_targets.flatten())

        total_loss = total_loss + loss.item()
        # backward pass
        loss.backward()

        # update weights
        optimizer.step()

        print(f"total loss: {total_loss/len(batch_inputs)} epoch: {epoch}")



total loss: 5.479189872741699 epoch: 0
total loss: 4.9501953125 epoch: 0
total loss: 4.65742301940918 epoch: 0
total loss: 4.6823039054870605 epoch: 0
total loss: 4.45601749420166 epoch: 0
total loss: 4.322627067565918 epoch: 0
total loss: 4.3138556480407715 epoch: 0
total loss: 4.028726577758789 epoch: 0
total loss: 3.991194248199463 epoch: 0
total loss: 3.384765625 epoch: 1
total loss: 3.473273277282715 epoch: 1
total loss: 3.1406948566436768 epoch: 1
total loss: 3.3235397338867188 epoch: 1
total loss: 3.1485495567321777 epoch: 1
total loss: 3.219804286956787 epoch: 1
total loss: 3.3095321655273438 epoch: 1
total loss: 3.2625160217285156 epoch: 1
total loss: 3.1792469024658203 epoch: 1
total loss: 2.883575916290283 epoch: 2
total loss: 2.9170165061950684 epoch: 2
total loss: 3.039325714111328 epoch: 2
total loss: 2.9221298694610596 epoch: 2
total loss: 2.821192502975464 epoch: 2
total loss: 3.075477361679077 epoch: 2
total loss: 2.838843822479248 epoch: 2
total loss: 2.61220932006835

In [21]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        #print(f'input: {idx_cond}')

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [26]:
inference_device = torch.device("mps")

model.to(inference_device)
model.eval()

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token("Every effort moves us", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_to_text(token_ids, tokenizer))

Output text:
 Every effort moves us had been through my surprise, for he answered with a deprecating laugh: "Yes--she's an awful simpleton
